In [ ]:
import sys; sys.path.append('../..')
import MeshFEM, mesh, sparse_matrices, benchmark, field_sampler, mesh_utilities
import inflatables_parametrization as parametrization, numpy as np, importlib, pickle, wall_generation
import utils
import py_newton_optimizer
from py_newton_optimizer import NewtonOptimizerOptions
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization, wall_width_formulas as wwf
from tri_mesh_viewer import TriMeshViewer

target_surf = mesh.Mesh('../../SiggraphExamples/Meshes/20200118_bike_helmet_v1_R00.obj')
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=True))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [ ]:
alphaMin = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(2, 10))
alphaMax = wwf.stretchFactorForCanonicalWallWidth(wwf.canonicalWallWidthForGeometry(1, 10))

In [ ]:
uv = utils.load('data/helmet_uv_2.pkl.gz')

In [ ]:
rparam = parametrization.RegularizedParametrizerSVD(target_surf, uv, transformForRigidMotionConstraint=False)

In [ ]:
benchmark.reset()

In [ ]:
nsubdiv=3
upsampledMesh, upsampledAngles, upsampledStretches = rparam.upsampledVertexLeftStretchAnglesAndMagnitudes(nsubdiv)
upsampledStretches = np.clip(upsampledStretches, alphaMin, alphaMax)
(sdfVertices, sdfTris, sdf) = wall_generation.evaluate_stripe_field(upsampledMesh.vertices(), upsampledMesh.triangles(), upsampledAngles,
                                                                    wwf.canonicalWallWidthForStretchFactor(upsampledStretches), frequency=0.2)

In [ ]:
pts, edges = wall_generation.extract_contours(sdfVertices, sdfTris, sdf,
                                              targetEdgeSpacing=4.0,
                                              minContourLen=10)

In [ ]:
# Optional manual cleanup (e.g., in Blender) when necessary:
# Remove vertices too close to neighboring contours (which cause many tiny triangles in the generated mesh and make the optimizer's job difficult).
# These can be detected by inspecting the wireframe visualization of the triangle mesh created below.
# A common issue is when fusing curves intersect the sheet boundary at a glancing angle; these fusing curves should be simplified to remove their vertices very close to the boundary.
#mesh.save('bad_contour.obj', pts, edges)
pts, edges = mesh.load_raw('medium_contours_cleaned.obj')

In [ ]:
triArea = 10.0

In [ ]:
import sheet_meshing
m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, pts, edges, triArea=triArea)

In [ ]:
mview = TriMeshViewer(m)
mview.showWireframe()
mview.show()

In [ ]:
import inflation
isheet = inflation.InflatableSheet(m, iwv)
uv = rparam.uv()

In [ ]:
print('sheet generation')
benchmark.report()
benchmark.reset()

In [ ]:
paramSampler = field_sampler.FieldSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

In [ ]:
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)

In [ ]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = False
targetAttractedSheet.fittingWeight = 0.0

In [ ]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-7
opts.niter = iterations_per_output

In [ ]:
import boundaries

In [ ]:
fixedVars = boundaries.getOuterBoundaryVars(isheet)

In [ ]:
# viewer = TriMeshViewer(isheet, width=768, height=640, wireframe=True)
# viewer.showWireframe()
# viewer.show()

In [ ]:
indicator = np.zeros(isheet.numVars() // 3)
indicator[np.array(fixedVars) // 3] = 1.0

In [ ]:
# viewer.update(scalarField=indicator)

In [ ]:
import time
isheet.pressure = 0.025
benchmark.reset()
for step in range(int(niter / iterations_per_output)):
    cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts)
    if cr.numIters() < iterations_per_output: break
    # viewer.update(False, isheet.visualizationMesh())
    time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
benchmark.report()

In [ ]:
print('initial inflation')
benchmark.report()
benchmark.reset()

In [ ]:
utils.save(targetAttractedSheet, 'data/helmet_medium_tas_init.pkl.gz')